# Détection de fraude bancaire avec PySpark

Traitement des données de transactions bancaires : lecture depuis HDFS, nettoyage, analyse,
modèles de classification et export des résultats.
Les cellules s'exécutent dans l'ordre (Restart du noyau, puis Run All).

In [1]:
import os
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, DoubleType

# Chemin du fichier dans HDFS (voir le README)
DATA_PATH = "hdfs://localhost:9000/data/raw/Bank_Transaction_Fraud_Detection.csv"

# Dossier où on enregistre les résultats pour Streamlit
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

# Démarrage de Spark
spark = SparkSession.builder.appName("fraude").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 08:43:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Étape 4.1 : lecture des données

Lecture du CSV depuis HDFS. Vérification : 24 colonnes et environ 200 000 lignes.

In [2]:
df_brut = spark.read.csv(DATA_PATH, header=True)

print("Nombre de colonnes :", len(df_brut.columns))
print("Nombre de lignes :", df_brut.count())

Nombre de colonnes : 24


Nombre de lignes : 200000


## Étape 4.2 : renommage et typage des colonnes

Renommage des colonnes en français, puis typage : nombres pour l'âge, les montants et la cible,
date pour la date de transaction, texte pour le reste.

In [3]:
# On crée un DataFrame avec les nouveaux noms (ancien nom -> nouveau nom)
df = df_brut.select(
    F.col("Customer_ID").alias("id_client"),
    F.col("Customer_Name").alias("nom_client"),
    F.col("Gender").alias("genre"),
    F.col("Age").alias("age"),
    F.col("State").alias("etat"),
    F.col("City").alias("ville"),
    F.col("Bank_Branch").alias("agence_bancaire"),
    F.col("Account_Type").alias("type_compte"),
    F.col("Transaction_ID").alias("id_transaction"),
    F.col("Transaction_Date").alias("date_transaction"),
    F.col("Transaction_Time").alias("heure_transaction"),
    F.col("Transaction_Amount").alias("montant"),
    F.col("Merchant_ID").alias("id_marchand"),
    F.col("Transaction_Type").alias("type_transaction"),
    F.col("Merchant_Category").alias("categorie_marchand"),
    F.col("Account_Balance").alias("solde_compte"),
    F.col("Transaction_Device").alias("appareil_transaction"),
    F.col("Transaction_Location").alias("localisation_transaction"),
    F.col("Device_Type").alias("type_appareil"),
    F.col("Is_Fraud").alias("est_fraude"),
    F.col("Transaction_Currency").alias("devise"),
    F.col("Customer_Contact").alias("contact_client"),
    F.col("Transaction_Description").alias("description_transaction"),
    F.col("Customer_Email").alias("email_client"),
)

# On change les types
df = df.withColumn("age", F.col("age").cast(IntegerType()))
df = df.withColumn("est_fraude", F.col("est_fraude").cast(IntegerType()))
df = df.withColumn("montant", F.col("montant").cast(DoubleType()))
df = df.withColumn("solde_compte", F.col("solde_compte").cast(DoubleType()))
df = df.withColumn("date_transaction", F.to_date(F.col("date_transaction"), "dd-MM-yyyy"))
# On garde seulement l'heure (les 2 premiers caractères de 16:04:07)
df = df.withColumn("heure_transaction", F.substring(F.col("heure_transaction"), 1, 2).cast(IntegerType()))

# Contrôle : si une valeur ne se convertit pas, elle devient vide (null)
print("age vides :", df.filter(F.col("age").isNull()).count())
print("montant vides :", df.filter(F.col("montant").isNull()).count())
print("date vides :", df.filter(F.col("date_transaction").isNull()).count())

age vides : 0


montant vides : 0


date vides : 0


## Étape 4.3 : statistiques descriptives

Statistiques de base, répartition de la cible, montants moyens, corrélations et taux de fraude par catégorie.

**Observations :**
1. Le taux de fraude est de 5,04 % (10 088 transactions frauduleuses sur 200 000).
2. Le montant moyen est presque identique selon la classe : 49 277,93 pour les fraudes, 49 551,83 pour les transactions normales.
3. Les corrélations entre les variables numériques et `est_fraude` sont proches de zéro (entre -0,0021 et 0,0001).

In [4]:
# Chiffres de base sur les colonnes numériques
df.select("age", "montant", "solde_compte", "est_fraude").describe().show()

# Combien de transactions normales et frauduleuses ?
nombre_total = df.count()
repartition = df.groupBy("est_fraude").count()
repartition = repartition.withColumn("pourcentage", F.round(F.col("count") / nombre_total * 100, 2))
repartition.show()

# Montant moyen des transactions normales et frauduleuses
df.groupBy("est_fraude").agg(F.round(F.avg("montant"), 2).alias("montant_moyen")).show()

# Corrélations : est-ce que les variables bougent ensemble avec la fraude ?
print("corrélation age / est_fraude :", round(df.stat.corr("age", "est_fraude"), 4))
print("corrélation montant / est_fraude :", round(df.stat.corr("montant", "est_fraude"), 4))
print("corrélation solde_compte / est_fraude :", round(df.stat.corr("solde_compte", "est_fraude"), 4))
print("corrélation montant / solde_compte :", round(df.stat.corr("montant", "solde_compte"), 4))

# Taux de fraude par type de transaction
df.groupBy("type_transaction").agg(
    F.count("*").alias("nombre"),
    F.round(F.avg("est_fraude") * 100, 2).alias("taux_fraude_pourcent"),
).orderBy(F.desc("taux_fraude_pourcent")).show()

26/11/19 08:44:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+------------------+------------------+-------------------+
|summary|              age|           montant|      solde_compte|         est_fraude|
+-------+-----------------+------------------+------------------+-------------------+
|  count|           200000|            200000|            200000|             200000|
|   mean|         44.01511|49538.015554200094|52437.988783549976|            0.05044|
| stddev|15.28877432696952| 28551.87400382295|27399.507128068493|0.21885165267877135|
|    min|               18|             10.29|           5000.82|                  0|
|    max|               70|          98999.98|          99999.95|                  1|
+-------+-----------------+------------------+------------------+-------------------+



+----------+------+-----------+
|est_fraude| count|pourcentage|
+----------+------+-----------+
|         1| 10088|       5.04|
|         0|189912|      94.96|
+----------+------+-----------+



+----------+-------------+
|est_fraude|montant_moyen|
+----------+-------------+
|         1|     49277.93|
|         0|     49551.83|
+----------+-------------+



corrélation age / est_fraude : -0.0015


corrélation montant / est_fraude : -0.0021
corrélation solde_compte / est_fraude : 0.0001


corrélation montant / solde_compte : -0.0017


+----------------+------+--------------------+
|type_transaction|nombre|taux_fraude_pourcent|
+----------------+------+--------------------+
|        Transfer| 39953|                5.19|
|          Credit| 40180|                 5.1|
|           Debit| 40050|                5.08|
|    Bill Payment| 40040|                4.93|
|      Withdrawal| 39777|                4.93|
+----------------+------+--------------------+



## Étape 4.4 : nettoyage

Création de `df_propre` à partir de `df`. Suppression de la colonne `devise` (valeur unique),
des données personnelles, des identifiants et du texte libre, puis des doublons, des lignes incomplètes
et des lignes incohérentes. Dans ce jeu de données, aucun doublon, aucune ligne incomplète et aucune ligne incohérente n'est trouvé.

In [5]:
# 1. La colonne devise est la même pour toutes les lignes : on la supprime
print("Devises différentes :", df.select("devise").distinct().count())

# 2. On supprime les doublons sur l'identifiant de transaction
df_propre = df.dropDuplicates(["id_transaction"])
print("Doublons supprimés :", df.count() - df_propre.count())

# 3. On supprime les données personnelles et les identifiants
df_propre = df_propre.drop(
    "devise", "nom_client", "contact_client", "email_client",
    "id_client", "id_transaction", "id_marchand", "description_transaction",
)

# 4. On supprime les lignes incomplètes
avant = df_propre.count()
df_propre = df_propre.dropna()
print("Lignes incomplètes supprimées :", avant - df_propre.count())

# 5. On supprime les lignes qui n'ont pas de sens
avant = df_propre.count()
df_propre = df_propre.filter(F.col("age") >= 18)
df_propre = df_propre.filter(F.col("age") <= 120)
df_propre = df_propre.filter(F.col("montant") > 0)
df_propre = df_propre.filter(F.col("solde_compte") >= 0)
print("Lignes incohérentes supprimées :", avant - df_propre.count())

print("Lignes restantes :", df_propre.count())

Devises différentes : 1


Doublons supprimés : 0


Lignes incomplètes supprimées : 0


Lignes incohérentes supprimées : 0


Lignes restantes : 200000


## Étape 4.5 : création des variables pour le modèle

Ajout du jour de la semaine, du mois, de la tranche horaire et du ratio montant / solde.
La cible `est_fraude` est copiée dans la colonne `label`.

In [6]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

# Nouvelles colonnes
df_features = df_propre.withColumn("jour_semaine", F.dayofweek(F.col("date_transaction")))
df_features = df_features.withColumn("mois", F.month(F.col("date_transaction")))

# Tranche horaire : nuit, matin, apres_midi ou soir
df_features = df_features.withColumn(
    "tranche_horaire",
    F.when(F.col("heure_transaction") < 6, "nuit")
     .when(F.col("heure_transaction") < 12, "matin")
     .when(F.col("heure_transaction") < 18, "apres_midi")
     .otherwise("soir"),
)

# Ratio : quelle part du solde représente le montant
df_features = df_features.withColumn(
    "ratio_montant_solde",
    F.when(F.col("solde_compte") > 0, F.col("montant") / F.col("solde_compte")).otherwise(0.0),
)

# La cible
df_features = df_features.withColumn("label", F.col("est_fraude").cast("double"))

print("Colonnes :", df_features.columns)

Colonnes : ['genre', 'age', 'etat', 'ville', 'agence_bancaire', 'type_compte', 'date_transaction', 'heure_transaction', 'montant', 'type_transaction', 'categorie_marchand', 'solde_compte', 'appareil_transaction', 'localisation_transaction', 'type_appareil', 'est_fraude', 'jour_semaine', 'mois', 'tranche_horaire', 'ratio_montant_solde', 'label']


## Étape 4.6 : machine learning

Découpage train / test (80 % / 20 %), encodage des variables texte et normalisation, puis entraînement
d'une régression logistique et d'une random forest. L'évaluation se fait sur le jeu de test.

**Résultats :**
- Régression logistique : AUC 0,4985.
- Random forest : AUC 0,5073.
- Les deux modèles ne prédisent aucune fraude (rappel de la classe fraude : 0).

Le F1 affiché (0,9261) est pondéré sur les deux classes : il reflète surtout la classe majoritaire.
Avec ces variables, les modèles ne séparent pas les transactions frauduleuses des transactions normales.

In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# 1. Découpage train / test
train, test = df_features.randomSplit([0.8, 0.2], seed=42)

# 2. Étapes de préparation : les textes deviennent des nombres
colonnes_texte = [
    "genre", "etat", "ville", "agence_bancaire", "type_compte",
    "type_transaction", "categorie_marchand", "appareil_transaction",
    "localisation_transaction", "type_appareil", "tranche_horaire",
]
etapes = []
for c in colonnes_texte:
    etapes.append(StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep"))

etapes.append(OneHotEncoder(
    inputCols=[c + "_idx" for c in colonnes_texte],
    outputCols=[c + "_oh" for c in colonnes_texte],
    handleInvalid="keep",
))

# Les colonnes à assembler : les textes encodés + les chiffres
colonnes_chiffres = ["age", "montant", "solde_compte", "heure_transaction",
                     "jour_semaine", "mois", "ratio_montant_solde"]
etapes.append(VectorAssembler(
    inputCols=[c + "_oh" for c in colonnes_texte] + colonnes_chiffres,
    outputCol="features_brut",
    handleInvalid="keep",
))
etapes.append(StandardScaler(inputCol="features_brut", outputCol="features", withMean=False, withStd=True))

# On apprend la préparation sur train seulement, puis on l'applique aux deux jeux
preparation = Pipeline(stages=etapes).fit(train)
train_pret = preparation.transform(train)
test_pret = preparation.transform(test)
print("Préparation terminée")

# Régression logistique
modele_lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
resultat_lr = modele_lr.fit(train_pret).transform(test_pret)

# Random forest
modele_rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=30, maxDepth=6, seed=42)
resultat_rf = modele_rf.fit(train_pret).transform(test_pret)

# Évaluation sur le jeu de test
evaluateur_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluateur_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

auc_lr = evaluateur_auc.evaluate(resultat_lr)
f1_lr = evaluateur_f1.evaluate(resultat_lr)
auc_rf = evaluateur_auc.evaluate(resultat_rf)
f1_rf = evaluateur_f1.evaluate(resultat_rf)

print("Régression logistique : AUC =", round(auc_lr, 4), "| F1 =", round(f1_lr, 4))
print("Random forest : AUC =", round(auc_rf, 4), "| F1 =", round(f1_rf, 4))

# Contrôle important : le F1 ci-dessus est pondéré sur les deux classes.
# On regarde donc la classe fraude (label = 1) à part.
evaluateur_rappel = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="recallByLabel", metricLabel=1.0)
evaluateur_f1_fraude = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="fMeasureByLabel", metricLabel=1.0)

print("Régression logistique : fraudes prédites =", resultat_lr.filter(F.col("prediction") == 1).count())
print("Régression logistique : rappel fraude =", round(evaluateur_rappel.evaluate(resultat_lr), 4), "| F1 fraude =", round(evaluateur_f1_fraude.evaluate(resultat_lr), 4))
print("Random forest : fraudes prédites =", resultat_rf.filter(F.col("prediction") == 1).count())
print("Random forest : rappel fraude =", round(evaluateur_rappel.evaluate(resultat_rf), 4), "| F1 fraude =", round(evaluateur_f1_fraude.evaluate(resultat_rf), 4))

Préparation terminée


26/11/19 08:46:28 WARN MemoryStore: Not enough space to cache rdd_576_2 in memory! (computed 30.3 MiB so far)
26/11/19 08:46:28 WARN BlockManager: Persisting block rdd_576_2 to disk instead.


Régression logistique : AUC = 0.4985 | F1 = 0.9261
Random forest : AUC = 0.5073 | F1 = 0.9261


Régression logistique : fraudes prédites = 0


Régression logistique : rappel fraude = 0.0 | F1 fraude = 0.0


Random forest : fraudes prédites = 0


Random forest : rappel fraude = 0.0 | F1 fraude = 0.0


## Étape 4.7 : export des résultats

Export de quelques tableaux au format CSV dans le dossier `outputs/`, utilisés par l'application Streamlit.

In [8]:
# Tableau des résultats des modèles
resultats = pd.DataFrame({
    "modele": ["Regression logistique", "Random forest"],
    "auc": [round(auc_lr, 4), round(auc_rf, 4)],
    "f1": [round(f1_lr, 4), round(f1_rf, 4)],
})
resultats.to_csv(OUT + "/resultats_modeles.csv", index=False)

# Chiffres de base
stats = df.select("age", "montant", "solde_compte", "est_fraude").describe().toPandas()
stats.to_csv(OUT + "/statistiques.csv", index=False)

# Répartition fraude / non fraude
repartition_pd = df.groupBy("est_fraude").count().toPandas()
repartition_pd.to_csv(OUT + "/repartition_fraude.csv", index=False)

print("Fichiers créés :", os.listdir(OUT))

Fichiers créés : ['statistiques.csv', 'repartition_fraude.csv', 'resultats_modeles.csv']
